In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# import numpy as np

In [ ]:
# Categorical features
CAT_FEATURES = [
    "education_level",
    "has_partner",
    "had_partner",
    "is_female",
    "ever_smoker",
    "is_current_smoker",
    "drinking_frequency",
]

EXCLUDED_FEATURES = ["survey_weight"]

TARGET = "has_diabetes_or_prediabetes"

In [ ]:
df = pd.read_csv("../dataset/output/processed_data_combined_metabolic_history_v2.csv")

In [ ]:
df.info()

In [ ]:
# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# Get numerical columns (excluding categorical, target, and excluded features)
all_excluded = CAT_FEATURES + EXCLUDED_FEATURES + [TARGET]
numerical_cols = [col for col in df.columns if col not in all_excluded]

In [ ]:
df[EXCLUDED_FEATURES].boxplot()

---

In [ ]:
PA_FEATURES = ["moderate_minutes_per_week", 
               "vigorous_minutes_per_week", 
               "sedentary_minutes_per_day"
]

SLEEP_FEATURES = ["sleep_hours_weekday", "sleep_hours_weekend"]

In [ ]:
df[PA_FEATURES].describe()

In [ ]:
df[SLEEP_FEATURES].describe()

---

## Distribution Comparison for Numerical Features

In [ ]:
# Define the sample weight column
SAMPLE_WEIGHT = "survey_weight"

# Get data for each target level
target_levels = df[TARGET].unique()

# Calculate number of rows needed (1 feature per row, 2 plots per feature)
n_features = len(numerical_cols)

# Create a single large figure with all numerical features
fig, axes = plt.subplots(n_features, 2, figsize=(16, 5 * n_features))

# Handle case where there's only one feature (axes won't be 2D)
if n_features == 1:
    axes = axes.reshape(1, -1)

for idx, col in enumerate(numerical_cols):
    # Left plot: Unweighted
    ax_unweighted = axes[idx, 0]
    for level in target_levels:
        data = df[df[TARGET] == level][col].dropna()
        ax_unweighted.hist(data, bins=30, alpha=0.5, label=f"{TARGET}={level}",
                          edgecolor="black", density=True)

    ax_unweighted.set_xlabel(col, fontsize=11)
    ax_unweighted.set_ylabel("Relative Frequency", fontsize=11)
    ax_unweighted.set_title(f"Unweighted: {col}", fontsize=12)
    ax_unweighted.legend(fontsize=10)

    # Right plot: Weighted
    ax_weighted = axes[idx, 1]
    for level in target_levels:
        mask = df[TARGET] == level
        data = df[mask][col].dropna()
        weights = df[mask].loc[data.index, SAMPLE_WEIGHT]

        ax_weighted.hist(data, bins=30, alpha=0.5, label=f"{TARGET}={level}",
                        edgecolor="black", density=True, weights=weights)

    ax_weighted.set_xlabel(col, fontsize=11)
    ax_weighted.set_ylabel("Weighted Relative Frequency", fontsize=11)
    ax_weighted.set_title(f"Weighted: {col}", fontsize=12)
    ax_weighted.legend(fontsize=10)

fig.suptitle(f"Distribution Comparison: Unweighted vs Weighted by {TARGET}",
             fontsize=16, y=0.998)
plt.tight_layout()
plt.show()

## Cross Tables for Categorical Features

In [ ]:
# Create visualizations for categorical features
n_cat_features = len(CAT_FEATURES)

# Create a single large figure with all categorical features
fig, axes = plt.subplots(n_cat_features, 2, figsize=(16, 5 * n_cat_features))

# Handle case where there's only one feature (axes won't be 2D)
if n_cat_features == 1:
    axes = axes.reshape(1, -1)

for idx, cat_col in enumerate(CAT_FEATURES):
    # Left plot: Stacked bar chart showing composition within each target level
    cross_tab = pd.crosstab(df[cat_col], df[TARGET])
    cross_tab_pct = cross_tab.div(cross_tab.sum(axis=0), axis=1) * 100

    cross_tab_pct.T.plot(kind="bar", stacked=True, ax=axes[idx, 0],
                         colormap="tab10", alpha=0.8, edgecolor="black")
    axes[idx, 0].set_xlabel(TARGET, fontsize=11)
    axes[idx, 0].set_ylabel("Percentage (%)", fontsize=11)
    axes[idx, 0].set_title(f"Composition of {cat_col} within each {TARGET} level", fontsize=12)
    axes[idx, 0].legend(title=cat_col, bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
    axes[idx, 0].set_xticklabels(axes[idx, 0].get_xticklabels(), rotation=0)
    axes[idx, 0].set_ylim(0, 100)

    # Right plot: Grouped bar chart showing target distribution within each feature level
    cross_tab_row_pct = cross_tab.div(cross_tab.sum(axis=1), axis=0) * 100

    cross_tab_row_pct.plot(kind="bar", ax=axes[idx, 1],
                           colormap="Set2", alpha=0.8, edgecolor="black", width=0.8)
    axes[idx, 1].set_xlabel(cat_col, fontsize=11)
    axes[idx, 1].set_ylabel("Percentage (%)", fontsize=11)
    axes[idx, 1].set_title(f"{TARGET} distribution within each {cat_col} level", fontsize=12)
    axes[idx, 1].legend(title=TARGET, fontsize=9)
    axes[idx, 1].set_xticklabels(axes[idx, 1].get_xticklabels(), rotation=45, ha="right")
    axes[idx, 1].set_ylim(0, 100)
    axes[idx, 1].axhline(y=50, color="gray", linestyle="--", linewidth=1, alpha=0.5)

fig.suptitle(f"Categorical Features vs {TARGET}", fontsize=16, y=0.998)
plt.tight_layout()
plt.show()

In [ ]:
df["has_diabetes_or_prediabetes"].value_counts()

In [ ]:
df["lab_positive"].value_counts()

In [ ]:
df["undiagnosed"].value_counts()

In [ ]:
sns.boxplot(y=df["phq9_score"])
plt.title("Box Plot of PHQ-9 Score")
plt.ylabel("phq9_score")
plt.show()

In [ ]:
df["phq9_score"].describe()

In [ ]:
# Histogram of sample weights (robust to column name typo)
candidates = [SAMPLE_WEIGHT, "survey_weigth", "survey_weight"]
sw_col = next((c for c in candidates if c in df.columns), None)
if sw_col is None:
    raise KeyError(f"No sample weight column found. Tried: {candidates}")

weights = df[sw_col].dropna()
plt.figure(figsize=(10, 4))
sns.histplot(weights, bins=50, edgecolor="black")
plt.xlabel(sw_col)
plt.ylabel("Count")
plt.title("Histogram of Sample Weights")
plt.tight_layout()
plt.show()

In [ ]:
# Compare target distribution: unweighted vs weighted
unweighted_pct = df[TARGET].value_counts(normalize=True).sort_index() * 100
weighted_pct = (
    df.groupby(TARGET)[SAMPLE_WEIGHT].sum().reindex(unweighted_pct.index)
    / df[SAMPLE_WEIGHT].sum()
    * 100
)

comparison = pd.DataFrame(
    {
        "Unweighted (%)": unweighted_pct,
        "Weighted (%)": weighted_pct,
    }
)

ax = comparison.plot(kind="bar", figsize=(8, 4), rot=0, edgecolor="black")
ax.set_xlabel(TARGET)
ax.set_ylabel("Percentage (%)")
ax.set_title(f"Unweighted vs Weighted Distribution of {TARGET}")
ax.legend(title="Version")
plt.tight_layout()
plt.show()

comparison

In [ ]:
df.shape

In [ ]:
df.info()

# LaTeX charts for Section 3.7

Generates the five figures used in Section 3.7 of the thesis. Each chart is saved as both PDF (vector, for `\includegraphics`) and PNG (300 dpi, fallback) under `./figures/`. All visible labels are in Spanish to match the LaTeX document.

Assumes `df` is loaded earlier in the notebook from `processed_data_combined_metabolic_history_v2.csv`.

In [ ]:
# ===== Shared setup: imports, rcParams, palette, age bands, save helper =====
from pathlib import Path

import matplotlib as mpl
import numpy as np

FIGURES_DIR = Path("./figures")
FIGURES_DIR.mkdir(exist_ok=True)

mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "axes.titleweight": "normal",
    "axes.titlepad": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.grid": True,
    "grid.color": "#E5E5E5",
    "grid.linewidth": 0.7,
    "axes.axisbelow": True,
})

# --- Palette ---
PRIMARY     = "#4C6E91"  # muted steel blue   — default fill, target=0
SECONDARY   = "#A05A5A"  # muted terracotta   — second category, target=1
ACCENT_WARM = "#C9A66B"  # warm sand          — single-bar highlight / callout
ACCENT_COOL = "#7A9B8A"  # sage               — tertiary if needed
GRID_DARK   = "#2D3E50"  # dark blue-grey     — text emphasis, subtitles
MISSING     = "#D5D5D5"  # light grey         — missing/NA segments

# Class-encoding aliases (used in charts 3, 4)
COLOR_NEG = PRIMARY
COLOR_POS = SECONDARY
COLOR_MISSING = MISSING

# --- Standard age bands ---
age_bins = [17, 29, 39, 49, 59, 69, 80]
age_labels = ["18–29", "30–39", "40–49", "50–59", "60–69", "70–80"]
df["age_band"] = pd.cut(df["RIDAGEYR"], bins=age_bins, labels=age_labels)


def save_fig(fig, name: str) -> None:
    """Save figure as PDF (vector) and PNG (300 dpi) with tight bbox and white background."""
    for ext in ("pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}",
                    bbox_inches="tight", facecolor="white")

In [ ]:
# ===== Chart 1: demographics (2x2 grid) =====
fig, axes = plt.subplots(2, 2, figsize=(10, 6.5))

# Panel 1: Age histogram (highlight top-coded bin)
ax = axes[0, 0]
ax.grid(axis="y"); ax.xaxis.grid(False)
ages = df["RIDAGEYR"]
bins = np.arange(15, 86, 5)
counts, _, patches = ax.hist(
    ages, bins=bins, color=PRIMARY,
    edgecolor="white", linewidth=0.8, rwidth=1.0,
)
patches[-1].set_facecolor(ACCENT_WARM)
top_h = counts[-1]
ax.annotate(
    "Edad ≥ 80\n(top-coding NHANES)",
    xy=(82, top_h), xytext=(80, top_h * 1.70),
    fontsize=8.5, color=GRID_DARK, ha="center",
    arrowprops={"arrowstyle": "->", "color": GRID_DARK, "lw": 0.8,
                "connectionstyle": "arc3,rad=-0.2"},
)
ax.set_xlabel("Edad (años)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de edad", loc="left", color=GRID_DARK)
ax.set_xlim(15, 86)
ax.set_ylim(0, max(counts) * 1.3)

# Panel 2: Sex distribution (inline white labels)
ax = axes[0, 1]
ax.grid(axis="x"); ax.yaxis.grid(False)
sex_counts = pd.Series({
    "Femenino":  int((df["is_female"] == 1).sum()),
    "Masculino": int((df["is_female"] == 0).sum()),
})
total = sex_counts.sum()
sex_pct = sex_counts / total * 100
bars = ax.barh(
    sex_counts.index, sex_counts.values,
    color=[SECONDARY, PRIMARY], height=0.55,
    edgecolor="white", linewidth=0.8,
)
for bar, pct in zip(bars, sex_pct.values):
    w = bar.get_width()
    ax.text(w - total * 0.02, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%", va="center", ha="right",
            color="white", fontsize=10, fontweight="medium")
ax.set_xlabel("Cantidad")
ax.set_title("Distribución por sexo", loc="left", color=GRID_DARK)
ax.set_xlim(0, total * 0.6)
ax.invert_yaxis()
ax.tick_params(axis="y", length=0)

# Panel 3: Education (drop level 0)
ax = axes[1, 0]
ax.grid(axis="y"); ax.xaxis.grid(False)
edu_map = {
    1: "< 9.° grado", 2: "9.°–11.°", 3: "Secundario",
    4: "Terciario\nincompleto", 5: "Universitario\no más",
}
edu_counts = (
    df["education_level"].value_counts().sort_index().drop(0.0, errors="ignore")
)
edu_pct = edu_counts / edu_counts.sum() * 100
edu_xticks = [edu_map[int(k)] for k in edu_counts.index]
bars = ax.bar(
    edu_xticks, edu_counts.values,
    color=PRIMARY, edgecolor="white", linewidth=0.8, width=0.65,
)
for bar, pct in zip(bars, edu_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + edu_counts.max() * 0.02,
            f"{pct:.1f}%", ha="center", va="bottom",
            fontsize=8.5, color=GRID_DARK)
ax.set_ylabel("Cantidad")
ax.set_title("Nivel educativo más alto alcanzado", loc="left", color=GRID_DARK)
ax.set_ylim(0, edu_counts.max() * 1.18)
ax.tick_params(axis="x", length=0)

# Panel 4: Marital status
ax = axes[1, 1]
ax.grid(axis="y"); ax.xaxis.grid(False)
mar_counts = pd.Series({
    "Con pareja":      int((df["has_partner"] == 1).sum()),
    "Tuvo pareja":     int((df["had_partner"] == 1).sum()),
    "Nunca en pareja": int(((df["has_partner"] == 0) & (df["had_partner"] == 0)).sum()),
})
mar_pct = mar_counts / len(df) * 100
bars = ax.bar(
    mar_counts.index, mar_counts.values,
    color=PRIMARY, edgecolor="white", linewidth=0.8, width=0.6,
)
for bar, pct in zip(bars, mar_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + mar_counts.max() * 0.02,
            f"{pct:.1f}%", ha="center", va="bottom",
            fontsize=8.5, color=GRID_DARK)
ax.set_ylabel("Cantidad")
ax.set_title("Estado civil", loc="left", color=GRID_DARK)
ax.set_ylim(0, mar_counts.max() * 1.18)
ax.tick_params(axis="x", length=0)

plt.tight_layout(pad=1.5, h_pad=2.5, w_pad=2.5)
save_fig(fig, "fig_3_7_1_demographics")
plt.close(fig)

In [ ]:
# ===== Chart 2: prevalence by age x sex =====
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.grid(axis="y"); ax.xaxis.grid(False)

prev_male, prev_female = [], []
for band in age_labels:
    m = df[(df["age_band"] == band) & (df["is_female"] == 0)]
    f = df[(df["age_band"] == band) & (df["is_female"] == 1)]
    prev_male.append(m["has_diabetes_or_prediabetes"].mean() * 100)
    prev_female.append(f["has_diabetes_or_prediabetes"].mean() * 100)

x = np.arange(len(age_labels))
width = 0.38

bars_m = ax.bar(x - 0.20, prev_male, width,
                color=PRIMARY, edgecolor="white", linewidth=0.8, label="Masculino")
bars_f = ax.bar(x + 0.20, prev_female, width,
                color=SECONDARY, edgecolor="white", linewidth=0.8, label="Femenino")
for bars in (bars_m, bars_f):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                f"{h:.1f}", ha="center", va="bottom", fontsize=8, color=GRID_DARK)

overall = df["has_diabetes_or_prediabetes"].mean() * 100
ax.axhline(overall, color=GRID_DARK, linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(len(age_labels) - 0.5, overall + 0.6,
        f"Prevalencia general: {overall:.1f}%",
        ha="right", va="bottom", fontsize=8.5, color=GRID_DARK, style="italic")

ax.set_xticks(x)
ax.set_xticklabels(age_labels)
ax.set_xlabel("Banda etaria")
ax.set_ylabel("Prevalencia (%)")
ax.set_title("Prevalencia del target por banda etaria y sexo",
             loc="left", color=GRID_DARK)
ax.legend(title="Sexo", loc="upper left", frameon=False)
ax.set_ylim(0, max(max(prev_male), max(prev_female)) * 1.22)
ax.tick_params(axis="x", length=0)

plt.tight_layout()
save_fig(fig, "fig_3_7_2_prevalence_age_sex")
plt.close(fig)

In [ ]:
# ===== Chart 3: lab status by target =====
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.grid(axis="x"); ax.yaxis.grid(False)

# Order: row 0 = negative (target=0), row 1 = positive (target=1).
# invert_yaxis() puts row 0 (negative) on top.
rows = [
    ("Autorreporte negativo (target = 0)", 0),
    ("Autorreporte positivo (target = 1)", 1),
]

for i, (label, t) in enumerate(rows):
    sub = df[df["has_diabetes_or_prediabetes"] == t]
    n = len(sub)
    p_pos  = (sub["lab_positive"] == 1).sum() / n * 100
    p_neg  = (sub["lab_positive"] == 0).sum() / n * 100
    p_miss = sub["lab_positive"].isna().sum() / n * 100

    ax.barh(i, p_pos, height=0.45, color=COLOR_POS,
            edgecolor="white", linewidth=0.8,
            label="Lab+ (HbA1c ≥ 5.7%)" if i == 0 else "")
    ax.barh(i, p_neg, left=p_pos, height=0.45, color=COLOR_NEG,
            edgecolor="white", linewidth=0.8,
            label="Lab− (HbA1c < 5.7%)" if i == 0 else "")
    ax.barh(i, p_miss, left=p_pos + p_neg, height=0.45, color=COLOR_MISSING,
            edgecolor="white", linewidth=0.8,
            label="Sin medición" if i == 0 else "")

    if p_pos >= 8:
        ax.text(p_pos / 2, i, f"{p_pos:.1f}%",
                ha="center", va="center", fontsize=9, color="white", fontweight="medium")
    if p_neg >= 8:
        ax.text(p_pos + p_neg / 2, i, f"{p_neg:.1f}%",
                ha="center", va="center", fontsize=9, color="white", fontweight="medium")
    if p_miss >= 8:
        ax.text(p_pos + p_neg + p_miss / 2, i, f"{p_miss:.1f}%",
                ha="center", va="center", fontsize=9, color=GRID_DARK)

ax.set_yticks(range(len(rows)))
ax.set_yticklabels([label for label, _ in rows])
ax.invert_yaxis()
ax.set_xlabel("Porcentaje (%)")
ax.set_xlim(0, 100)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_title("Estado de laboratorio (HbA1c) por clase del autorreporte",
             loc="left", color=GRID_DARK)
ax.tick_params(axis="y", length=0)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35),
          ncol=3, frameon=False)

plt.tight_layout()
save_fig(fig, "fig_3_7_3_lab_status_by_target")
plt.close(fig)

In [ ]:
# ===== Chart 4: features by class (2x4 grid) =====
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
axes_flat = axes.flatten()

# 5 continuous features as box plots
box_panels = [
    ("RIDAGEYR",    "Edad",         "Edad (años)"),
    ("BMXBMI",      "IMC",          "IMC (kg/m²)"),
    ("BMXWAIST",    "Cintura",      "Cintura (cm)"),
    ("systolic_bp", "PA sistólica", "PA sistólica (mmHg)"),
    ("phq9_score",  "PHQ-9",        "PHQ-9"),
]

# 3 rate-style bars. value_fn returns the per-class rate (0-100).
# moderate_minutes_per_week → % with any moderate activity (continuous → binary framing
# because Q1/median/Q3 collapse near zero make a boxplot unreadable).
bar_panels = [
    ("moderate_minutes_per_week", "Actividad mod.", "% con actividad > 0 min/sem",
     lambda s: (s > 0).mean() * 100),
    ("told_high_bp", "Hipertensión", "Hipertensión autorrep. (%)",
     lambda s: s.mean() * 100),
    ("told_high_cholesterol", "Colesterol", "Colesterol autorrep. (%)",
     lambda s: s.mean() * 100),
]

TITLE_KW = {"loc": "left", "color": GRID_DARK, "fontsize": 10}

for ax, (col, title, ylabel) in zip(axes_flat[:5], box_panels):
    ax.grid(axis="y"); ax.xaxis.grid(False)
    data_neg = df[df["has_diabetes_or_prediabetes"] == 0][col].dropna()
    data_pos = df[df["has_diabetes_or_prediabetes"] == 1][col].dropna()

    bp = ax.boxplot(
        [data_neg, data_pos],
        showfliers=False, patch_artist=True, widths=0.5,
        medianprops={"color": "white", "linewidth": 1.5},
    )
    for patch, color in zip(bp["boxes"], [COLOR_NEG, COLOR_POS]):
        patch.set_facecolor(color)
        patch.set_edgecolor(color)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Neg.", "Pos."])
    ax.set_ylabel(ylabel)
    ax.set_title(title, **TITLE_KW)
    ax.tick_params(axis="x", length=0)

for ax, (col, title, ylabel, value_fn) in zip(axes_flat[5:], bar_panels):
    ax.grid(axis="y"); ax.xaxis.grid(False)
    series_neg = df[df["has_diabetes_or_prediabetes"] == 0][col].dropna()
    series_pos = df[df["has_diabetes_or_prediabetes"] == 1][col].dropna()
    rate_neg = value_fn(series_neg)
    rate_pos = value_fn(series_pos)
    bars = ax.bar([0, 1], [rate_neg, rate_pos],
                  color=[COLOR_NEG, COLOR_POS],
                  edgecolor="white", linewidth=0.8, width=0.55)
    for bar, val in zip(bars, [rate_neg, rate_pos]):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 1.8,
                f"{val:.1f}%", ha="center", va="bottom",
                fontsize=8.5, color=GRID_DARK)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Neg.", "Pos."])
    ax.set_ylabel(ylabel)
    ax.set_title(title, **TITLE_KW)
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", length=0)

legend_handles = [
    Patch(facecolor=COLOR_NEG, edgecolor="white", linewidth=0.8, label="Negativo"),
    Patch(facecolor=COLOR_POS, edgecolor="white", linewidth=0.8, label="Positivo"),
]
fig.legend(handles=legend_handles, loc="upper center", ncol=2,
           frameon=False, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout(pad=1.5, h_pad=2.5, w_pad=2.5)
fig.subplots_adjust(top=0.92)
save_fig(fig, "fig_3_7_4_features_by_class")
plt.close(fig)

In [ ]:
# ===== Chart 5: Spearman correlation heatmap =====
FEATURES_24 = [
    "RIDAGEYR", "education_level", "has_partner", "had_partner", "is_female",
    "BMXBMI", "BMXWAIST", "BMXWT", "BMXHT",
    "systolic_bp", "diastolic_bp",
    "told_high_bp", "told_high_cholesterol",
    "ever_smoker", "is_current_smoker",
    "drinking_frequency", "drinks_per_day", "binge_episodes_month",
    "sleep_hours_weekday", "sleep_hours_weekend",
    "phq9_score",
    "moderate_minutes_per_week", "vigorous_minutes_per_week", "sedentary_minutes_per_day",
]

FEATURE_LABELS_ES = {
    "RIDAGEYR": "Edad",
    "education_level": "Nivel educativo",
    "has_partner": "Con pareja",
    "had_partner": "Tuvo pareja",
    "is_female": "Es mujer",
    "BMXBMI": "IMC",
    "BMXWAIST": "Cintura",
    "BMXWT": "Peso",
    "BMXHT": "Altura",
    "systolic_bp": "PA sistólica",
    "diastolic_bp": "PA diastólica",
    "told_high_bp": "Hipertensión autorrep.",
    "told_high_cholesterol": "Colesterol autorrep.",
    "ever_smoker": "Fumador alguna vez",
    "is_current_smoker": "Fumador actual",
    "drinking_frequency": "Frec. consumo alcohol",
    "drinks_per_day": "Bebidas por día",
    "binge_episodes_month": "Episodios binge/mes",
    "sleep_hours_weekday": "Sueño semana",
    "sleep_hours_weekend": "Sueño fin de semana",
    "phq9_score": "PHQ-9",
    "moderate_minutes_per_week": "Actividad moderada",
    "vigorous_minutes_per_week": "Actividad vigorosa",
    "sedentary_minutes_per_day": "Sedentarismo",
}


def _cell_text_color(value: float) -> str:
    """Black for low |ρ|, white once contrast against RdBu_r requires it."""
    return "white" if abs(value) >= 0.5 else "black"


corr = df[FEATURES_24].corr(method="spearman")
labels_es = [FEATURE_LABELS_ES[c] for c in FEATURES_24]
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
corr_masked = corr.mask(mask)

fig, ax = plt.subplots(figsize=(9.5, 9))
ax.grid(False)
im = ax.imshow(corr_masked, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")

for i in range(len(FEATURES_24)):
    for j in range(len(FEATURES_24)):
        if i == j:
            ax.text(j, i, "—", ha="center", va="center", fontsize=7, color="black")
        elif j < i:
            v = corr.iloc[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=7, color=_cell_text_color(v))

ax.set_xticks(range(len(FEATURES_24)))
ax.set_yticks(range(len(FEATURES_24)))
ax.set_xticklabels(labels_es, rotation=45, ha="right")
ax.set_yticklabels(labels_es)

# White separator lines after each conceptual group (thick enough to read as gutters)
group_separators = [4, 8, 10, 12, 14, 17, 19, 20]
for s in group_separators:
    ax.axhline(s + 0.5, color="white", linewidth=2.5)
    ax.axvline(s + 0.5, color="white", linewidth=2.5)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(top=False, right=False, bottom=False, left=False)

cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, shrink=0.7)
cbar.set_label("Correlación de Spearman (ρ)")

plt.tight_layout()
save_fig(fig, "fig_3_7_5_correlation_heatmap")
plt.close(fig)

In [ ]:
from IPython.display import Image, display

for name in [
    "fig_3_7_1_demographics",
    "fig_3_7_2_prevalence_age_sex",
    "fig_3_7_3_lab_status_by_target",
    "fig_3_7_4_features_by_class",
    "fig_3_7_5_correlation_heatmap",
]:
    print(name)
    display(Image(filename=str(FIGURES_DIR / f"{name}.png")))